In [0]:
#Leitura camada bronze
df_associado = spark.table("bronze.associado_diaria_05hs")

In [0]:
from pyspark.sql import functions as F

df_silver_associado = (
  df_associado
  .select(
    F.col("id").cast("long").alias("id"),
    F.initcap(F.trim(F.col("nome"))).alias("nome"),
    F.initcap(F.trim(F.col("sobrenome"))).alias("sobrenome"),
    F.col("idade").cast("int").alias("idade"),
    F.lower(F.trim(F.col("email"))).alias("email"),
    F.col("data_nascimento").cast("date").alias("data_nascimento"),
    F.current_timestamp().alias("dt_ingestao_silver")
  )
  .dropDuplicates(["id"])
)


In [0]:
# Qualidade dos dados
erros = []  # Lista para armazenar erros de qualidade identificados

# Verifica se existe algum registro com id nulo
if df_silver_associado.filter(F.col("id").isNull()).count() > 0: 
    erros.append("id nulo")

# Verifica se existe algum registro com idade fora do intervalo permitido (menor que 0 ou maior que 120)
if df_silver_associado.filter((F.col("idade") < 0) | (F.col("idade") > 120)).count() > 0: 
    erros.append("idade inválida")

# Verifica se existe algum registro com email nulo
if df_silver_associado.filter(F.col("email").isNull()).count() > 0: 
    erros.append("email nulo")

# Se houver erros, imprime cada erro e lança uma exceção caso necessario
if erros:
    for erro in erros:
        print(f"Falha qualidade associado_silver: {erro}")
    raise Exception("Falha qualidade associado_silver: " + " | ".join(erros))

In [0]:
#Salvando base na camada silver e modo overwrite
(df_silver_associado.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("silver.associado_diaria_05hs"))